# DRI-19 Run #4 Vision-LoRA Ablation

Clean Colab ablation against run #3. This keeps the run #3 recipe fixed and changes only the LoRA target scope by adding adapters to Qwen2-VL vision attention modules.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri-19-run4-vision-lora"  # switch to main after this branch is merged
WORKDIR = Path("/content/Drishti")

RUN_NAME = "drishti-qlora-run4-vision-lora-ablation"
OUTPUT_DIR = Path("outputs/dri19-run4-vision-lora-ablation")
EVAL_ROOT = Path("outputs/eval/dri19-run4-vision-lora-ablation")
HF_REPO_ID = "ShivSingh123/drishti-qlora-run4-vision-lora-ablation"

SEED = 42
DIAGNOSTIC_STEPS = [400, 800]
CONTINUATION_STEPS = [1600, 3000, 4950]
VAL_EVAL_PER_CLASS = 150

RUN3_BASELINE = {
    "accuracy": 0.942222,
    "macro_f1": 0.910004,
    "per_class_f1": {
        "active_tb": 0.809195,
        "healthy": 0.954399,
        "sick_but_non_tb": 0.966418,
    },
}
METRIC_TIE_TOLERANCE = 0.005
PER_CLASS_DROP_TOLERANCE = {
    "active_tb": 0.04,
    "healthy": 0.02,
    "sick_but_non_tb": 0.02,
}
VISION_FALLBACK_ARGS: list[str] = []

In [ ]:
if not WORKDIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {WORKDIR}
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

def require_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab secret: {name}")
    return value

os.environ["KAGGLE_USERNAME"] = require_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = require_secret("KAGGLE_KEY")
os.environ["WANDB_API_KEY"] = require_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = require_secret("HF_TOKEN")

login(token=os.environ["HF_TOKEN"])
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

In [ ]:
import importlib.metadata as metadata
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU before continuing.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu={gpu_name} vram_gb={gpu_mem_gb:.1f}")
print(f"torch={torch.__version__} cuda={torch.version.cuda}")
for package in ("transformers", "peft", "bitsandbytes", "accelerate", "wandb"):
    print(f"{package}={metadata.version(package)}")

preferred = any(name in gpu_name for name in ("A100", "L4"))
if not preferred:
    if "T4" in gpu_name:
        raise RuntimeError("T4 detected. Use this notebook only for smoke/setup checks; reconnect until Colab gives A100 or L4.")
    raise RuntimeError(f"Unexpected GPU for run #4: {gpu_name}. Reconnect for A100 or L4.")

In [ ]:
!python download_dataset.py
!python generate_jsonl.py --output-dir data/processed

import json
from collections import Counter

expected = {
    "train": Counter({"active_tb": 600, "healthy": 3000, "sick_but_non_tb": 3000}),
    "val": Counter({"active_tb": 200, "healthy": 800, "sick_but_non_tb": 800}),
}

for split, expected_counts in expected.items():
    counts = Counter()
    with open(f"data/processed/{split}.jsonl", encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            assistant = payload["messages"][1]["content"]
            assert assistant.startswith("Classification: "), assistant
            assert assistant.count("\n") == 0, assistant
            counts[assistant.removeprefix("Classification: ")] += 1
    print(split, dict(sorted(counts.items())))
    assert counts == expected_counts, (split, counts, expected_counts)

In [ ]:
import subprocess

def run(command: list[str], *, capture: bool = False) -> subprocess.CompletedProcess | None:
    print("\n$ " + " ".join(command))
    if capture:
        result = subprocess.run(command, text=True, capture_output=True)
        print(result.stdout)
        print(result.stderr)
        return result
    subprocess.run(command, check=True)
    return None

run(["python", "-m", "unittest", "-v"])
run(["python", "train_qlora.py", "--dry-run", "--train-limit", "2", "--eval-limit", "2"])

VISION_FALLBACK_ARGS: list[str] = []
setup_cmd = [
    "python", "train_qlora.py",
    "--setup-only",
    "--setup-backward-check",
    "--train-limit", "2",
    "--eval-limit", "2",
    "--wandb-mode", "disabled",
    "--output-dir", "outputs/dri19-setup-validation",
    "--rank", "32",
    "--alpha", "64",
    "--lora-dropout", "0.05",
    "--lora-target-scope", "language-vision-attn",
    "--seed", str(SEED),
]
result = run(setup_cmd, capture=True)
if result and result.returncode != 0:
    combined_output = f"{result.stdout}\n{result.stderr}".lower()
    if "out of memory" not in combined_output and "cuda oom" not in combined_output:
        raise RuntimeError("Setup backward check failed for a non-OOM reason.")
    print("Vision rank 32 OOM during setup check; retrying with vision rank 16 / alpha 32.")
    VISION_FALLBACK_ARGS = ["--vision-rank", "16", "--vision-alpha", "32"]
    fallback = run(setup_cmd + VISION_FALLBACK_ARGS, capture=True)
    if fallback and fallback.returncode != 0:
        raise RuntimeError("Run #4 vision LoRA setup is infeasible on this GPU, even with vision rank 16.")

print(f"VISION_FALLBACK_ARGS={VISION_FALLBACK_ARGS}")

In [ ]:
os.environ["WANDB_PROJECT"] = "tbx11k-qwen-vl-finetuning"
os.environ["WANDB_MODE"] = "online"
os.environ["WANDB_TAGS"] = "dri-19,run4,colab,vision-lora,ablation,soft-balanced"
os.environ["WANDB_NOTES"] = "Run #4 clean ablation vs run #3: same seed/data/sampler/hparams, adds Qwen2-VL vision attention LoRA targets."
os.environ["WANDB_RUN_ID"] = "dri19-run4-vision-lora-ablation"
os.environ["WANDB_RESUME"] = "allow"

def upload_path(local_path: Path, repo_path: str) -> None:
    if not local_path.exists():
        raise FileNotFoundError(local_path)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(local_path),
        path_in_repo=repo_path,
    )

def adapter_path_for_step(step: int) -> Path:
    checkpoint = OUTPUT_DIR / f"checkpoint-{step}"
    return checkpoint if checkpoint.exists() else OUTPUT_DIR

base_train_cmd = [
    "python", "train_qlora.py",
    "--run-name", RUN_NAME,
    "--output-dir", str(OUTPUT_DIR),
    "--rank", "32",
    "--alpha", "64",
    "--lora-dropout", "0.05",
    "--lora-target-scope", "language-vision-attn",
    "--lr", "1.5e-4",
    "--epochs", "3",
    "--batch-size", "1",
    "--grad-accum", "4",
    "--warmup-steps", "150",
    "--weight-decay", "0.01",
    "--lr-scheduler-type", "cosine",
    "--sampling-strategy", "soft-balanced",
    "--save-steps", "200",
    "--eval-steps", "200",
    "--logging-steps", "10",
    "--seed", str(SEED),
] + VISION_FALLBACK_ARGS

print(" ".join(base_train_cmd))

In [ ]:
previous_checkpoint = None
for target_step in DIAGNOSTIC_STEPS:
    train_cmd = base_train_cmd + ["--max-steps", str(target_step)]
    if previous_checkpoint is not None:
        train_cmd += ["--resume-from-checkpoint", str(previous_checkpoint)]
    run(train_cmd)

    adapter_dir = adapter_path_for_step(target_step)
    eval_dir = EVAL_ROOT / f"checkpoint-{target_step}-diagnostic"
    try:
        run([
            "python", "evaluate_checkpoint.py",
            "--adapter-dir", str(adapter_dir),
            "--data-dir", "data/processed",
            "--split", "val",
            "--output-dir", str(eval_dir),
            "--batch-size", "3",
            "--limit-per-class", str(VAL_EVAL_PER_CLASS),
            "--gate", "run4-diagnostic",
            "--fail-on-gate-fail",
        ])
    finally:
        upload_path(adapter_dir, f"checkpoints/checkpoint-{target_step}")
        if eval_dir.exists():
            upload_path(eval_dir, f"eval/checkpoint-{target_step}-diagnostic")

    previous_checkpoint = adapter_dir

print("Diagnostic checkpoints passed. Continue only if W&B and eval distribution look sane.")

In [ ]:
if previous_checkpoint is None:
    raise RuntimeError("Run the diagnostic cell first.")

for target_step in CONTINUATION_STEPS:
    train_cmd = base_train_cmd + ["--max-steps", str(target_step), "--resume-from-checkpoint", str(previous_checkpoint)]
    run(train_cmd)

    adapter_dir = adapter_path_for_step(target_step)
    eval_dir = EVAL_ROOT / f"checkpoint-{target_step}-full-val"
    try:
        run([
            "python", "evaluate_checkpoint.py",
            "--adapter-dir", str(adapter_dir),
            "--data-dir", "data/processed",
            "--split", "val",
            "--output-dir", str(eval_dir),
            "--batch-size", "3",
            "--gate", "run3-full",
            "--fail-on-gate-fail",
        ])
    finally:
        upload_path(adapter_dir, f"checkpoints/checkpoint-{target_step}")
        if eval_dir.exists():
            upload_path(eval_dir, f"eval/checkpoint-{target_step}-full-val")

    previous_checkpoint = adapter_dir

print("Run #4 continuation complete.")

In [ ]:
import json

def load_eval(step: int) -> dict:
    path = EVAL_ROOT / f"checkpoint-{step}-full-val" / "eval_results.json"
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))

def print_scores(step: int, metrics: dict) -> None:
    print(f"\n=== checkpoint-{step}-full-val ===")
    print(f"accuracy: {metrics['accuracy']:.6f}")
    print(f"macro_f1: {metrics['macro_f1']:.6f}")
    print(f"prediction_distribution: {metrics['prediction_distribution']}")
    print(f"gate_passed: {metrics.get('gate', {}).get('passed')}")
    for label in ("active_tb", "healthy", "sick_but_non_tb"):
        cls = metrics["per_class"][label]
        print(f"{label}: precision={cls['precision']:.6f} recall={cls['recall']:.6f} f1={cls['f1']:.6f} support={cls['support']}")
    print("confusion_matrix:")
    print(json.dumps(metrics["confusion_matrix"], indent=2))

full_results = {}
for step in CONTINUATION_STEPS:
    try:
        full_results[step] = load_eval(step)
        print_scores(step, full_results[step])
    except FileNotFoundError as exc:
        print(f"MISSING: {exc}")

if full_results:
    best_step, best_metrics = max(full_results.items(), key=lambda item: item[1]["macro_f1"])
    accuracy_delta = best_metrics["accuracy"] - RUN3_BASELINE["accuracy"]
    macro_delta = best_metrics["macro_f1"] - RUN3_BASELINE["macro_f1"]
    per_class_ok = True
    print(f"\n=== best checkpoint by macro-F1: {best_step} ===")
    print(f"accuracy_delta_vs_run3: {accuracy_delta:.6f}")
    print(f"macro_f1_delta_vs_run3: {macro_delta:.6f}")
    for label, baseline_f1 in RUN3_BASELINE["per_class_f1"].items():
        actual_f1 = best_metrics["per_class"][label]["f1"]
        drop = baseline_f1 - actual_f1
        allowed = PER_CLASS_DROP_TOLERANCE[label]
        passed = drop <= allowed
        per_class_ok = per_class_ok and passed
        print(f"{label}_f1_delta_vs_run3: {actual_f1 - baseline_f1:.6f} allowed_drop={allowed:.2f} passed={passed}")
    positive_ablation = (
        accuracy_delta > METRIC_TIE_TOLERANCE
        and macro_delta > METRIC_TIE_TOLERANCE
        and best_metrics.get("gate", {}).get("passed") is True
        and per_class_ok
    )
    print(f"positive_ablation={positive_ablation}")